# Synthea COVID-19 Module Analysis
This notebook provides and analysis of data generated by Synthea's COVID-19 module. Analysis is run on the CSV output from Synthea.

Exploring the synthetic data with duckdb

COVID-19
COVID-19 10K, CSV | [mirror]: 54 MB
Ten thousand synthetic patients records with COVID-19 in the CSV format.
COVID-19 100K, CSV: 512 MB
One hundred thousand synthetic patients records with COVID-19 in the CSV format.
Please cite the SyntheaTM COVID-19 data as:

Walonoski J, Klaus S, Granger E, Hall D, Gregorowicz A, Neyarapally G, Watson A, Eastman J. Synthea™ Novel coronavirus (COVID-19) model and synthetic data set. Intelligence-Based Medicine. 2020 Nov;1:100007. https://doi.org/10.1016/j.ibmed.2020.100007

In [ ]:
# download 10k synthetic data
# !wget https://mitre.box.com/shared/static/9iglv8kbs1pfi7z8phjl9sbpjk08spze.zip
# !unzip 9iglv8kbs1pfi7z8phjl9sbpjk08spze.zip
# !mv -v 10k_synthea_covid19_csv/* .

# download 100k synthetic data
!wget https://mitre.box.com/shared/static/wk3560f962ozlg7sd2oj1zxk73ayqvm0.zip
!unzip wk3560f962ozlg7sd2oj1zxk73ayqvm0.zip
!mv -v 100k_synthea_covid19_csv/* .

In [ ]:
import glob
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import datetime
from pathlib import Path
from typing import Tuple
import numpy as np

In [ ]:
def load_table(connection: duckdb.DuckDBPyConnection, table: str, path: str) -> None:
    con.execute(
        f'CREATE OR REPLACE TABLE "{table}" AS SELECT * FROM read_csv_auto({path}, HEADER=TRUE);'
    )

In [ ]:
# glob * csv into a list named csv_files

csv_files = glob.glob('*.csv')
csv_files

In [ ]:
# Connect to a local DuckDB database
con = duckdb.connect('10k_synthea_covid19.db')

In [ ]:
# # Ingest Single csv
# tablename = 'immunizations'
# file = 'immunizations.csv'
# load_table(con, tablename, Path(file))

In [ ]:
! du -hsc *.csv

On a system with 12.7 Gb system ram takes about 1 min to load 4.8 Gb of csv

In [ ]:
for file in csv_files:
    tablename = file.split('.')[0]
    print(f"Loading {tablename}")
    load_table(con,tablename, file)

In [ ]:
# Grab the IDs of patients that have been diagnosed with COVID-19
result = con.execute("""
SELECT DISTINCT PATIENT
FROM conditions
WHERE CODE = '840539006'
""").fetchall()

covid_patient_ids_duckdb = [row[0] for row in result]

print(covid_patient_ids_duckdb)

In [ ]:
print(f"Number of patients with COVID-19: {len(covid_patient_ids_duckdb)}")

# Patients with negative test results
This grabs every patient with a negative SARS-CoV-2 test. This will include patients who tested negative up front as well as patients that tested negative after leaving the hospital

In [ ]:
# DuckDB SQL to find unique patient IDs with a negative COVID-19 observation
result_negative = con.execute("""
SELECT DISTINCT PATIENT
FROM observations
WHERE CODE = '94531-1' AND VALUE = 'Not detected (qualifier value)'
""").fetchall()


Grabs IDs for all patients that died in the simulation. This will be more than just COVID-19 deaths.

In [ ]:
# DuckDB SQL to find deceased patients
result_deceased = con.execute("""
SELECT Id
FROM patients
WHERE DEATHDATE IS NOT NULL
""").fetchall()

deceased_patient_ids_duckdb = [row[0] for row in result_deceased]

print(deceased_patient_ids_duckdb)

In [ ]:
print(f"Number of deceased patients: {len(deceased_patient_ids_duckdb)}")

In [ ]:
#  DuckDB SQL to find patients who completed isolation
result_completed_isolation = con.execute("""
SELECT PATIENT
FROM careplans
WHERE CODE = '736376001'
  AND STOP IS NOT NULL
  AND REASONCODE = '840539006'
""").fetchall()

completed_isolation_patient_ids_duckdb = [row[0] for row in result_completed_isolation]

print(completed_isolation_patient_ids_duckdb)

In [ ]:
print(f"Number of patients who completed isolation: {len(completed_isolation_patient_ids_duckdb)}")

In [ ]:
survivor_ids = np.union1d(completed_isolation_patient_ids_duckdb, result_negative)

In [ ]:
len(survivor_ids)

In [ ]:
# Grab IDs for patients with admission due to COVID-19
result_inpatient = con.execute("""
SELECT PATIENT
FROM encounters
WHERE REASONCODE = '840539006' AND CODE = '1505002'
""").fetchall()

inpatient_ids_duckdb = [row[0] for row in result_inpatient]

print(inpatient_ids_duckdb)

In [ ]:
print(f"Number of patients who were inpatients: {len(inpatient_ids_duckdb)}")

In [ ]:
# The number of inpatient survivors
np.intersect1d(inpatient_ids_duckdb, survivor_ids).shape

In [ ]:
# The number of inpatient non-survivors
np.intersect1d(inpatient_ids_duckdb, deceased_patient_ids_duckdb).shape

In [ ]:
# ! wget https://raw.githubusercontent.com/kevin3/module-validation/refs/heads/master/notebooks/analysis.py
import analysis

In [ ]:
!ls -lh conditions*
conditions = pd.read_csv("./conditions.csv") #refactor

# conditions = con.execute("""
# SELECT *
# FROM conditions
# """).fetchall()

In [ ]:
conditions

In [ ]:
analysis.outcome_table(inpatient_ids_duckdb, survivor_ids, deceased_patient_ids_duckdb, conditions)

In [ ]:
lab_obs_result = con.execute("""
SELECT *
FROM observations
WHERE CODE IN ('48065-7', '26881-3', '2276-4', '89579-7', '2532-0', '731-0', '14804-9')
""").fetchall()

# Convert the result to a pandas DataFrame with the correct number of columns
lab_obs = pd.DataFrame(lab_obs_result, columns=["DATE", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "VALUE", "UNITS", "TYPE"]) # Added 'TYPE' column

In [ ]:
lab_obs

In [ ]:
lab_obs_result

In [ ]:
# Select COVID-19 conditions out of all conditions in the simulation

# covid_conditions = conditions[conditions.CODE == 840539006]

covid_conditions = con.execute("""
SELECT *
FROM conditions
WHERE CODE = '840539006'
""").fetchall

In [ ]:
# Merge the COVID-19 conditions with the patients

# covid_patients = covid_conditions.merge(patients, how='left', left_on='PATIENT', right_on='Id')



In [ ]:
# Execute a DuckDB query to left join covid_conditions with the patients table
covid_patients  = con.execute("""
SELECT cc.*, p.BIRTHDATE, p.GENDER, p.RACE, p.ETHNICITY, p.MARITAL, p.ADDRESS, p.CITY, p.STATE, p.ZIP, p.LAT, p.LON, p.HEALTHCARE_EXPENSES, p.HEALTHCARE_COVERAGE
FROM conditions AS cc
LEFT JOIN patients AS p ON cc.PATIENT = p.Id
WHERE cc.CODE = '840539006'
""").fetchall()

# Convert the result to a pandas DataFrame for easier manipulation and display
covid_patients  = pd.DataFrame(covid_patients , columns=["START", "STOP", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "BIRTHDATE", "GENDER", "RACE", "ETHNICITY", "MARITAL", "ADDRESS", "CITY", "STATE", "ZIP", "LAT", "LON", "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE"])

# Display the first few rows of the joined DataFrame
display(covid_patients.head())

In [ ]:
# Add an attribute to the DataFrame indicating whether this is a survivor or not.

covid_patients['survivor'] = covid_patients.PATIENT.isin(survivor_ids)

In [ ]:
# Reduce the columns on the DataFrame to ones needed

covid_patients = covid_patients[['START', 'PATIENT', 'survivor', 'CODE']]
covid_patients

In [ ]:
# Calculate attributes needed to support the plot. Also coerce all lab values into a numeric data type.

covid_patients_obs = covid_patients.merge(lab_obs, on='PATIENT')
covid_patients_obs['START'] = pd.to_datetime(covid_patients_obs.START, utc=True)
covid_patients_obs['DATE'] = pd.to_datetime(covid_patients_obs.DATE)
# covid_patients_obs['lab_days'] = covid_patients_obs.DATE - covid_patients_obs.START
# covid_patients_obs['days'] = np.around(covid_patients_obs.lab_days / np.timedelta64(1, 'D'))
# covid_patients_obs['VALUE'] = pd.to_numeric(covid_patients_obs['VALUE'], errors='coerce')

In [ ]:
covid_patients_obs.START

In [ ]:
covid_patients_obs['START'] = covid_patients_obs['START'].dt.tz_localize(None)

In [ ]:
covid_patients_obs.DATE

In [ ]:
covid_patients_obs['DATE'] = pd.to_datetime(covid_patients_obs.DATE)

In [ ]:
#after conver to timezone-naive format for above
covid_patients_obs['lab_days'] = covid_patients_obs.DATE - covid_patients_obs.START

In [ ]:
covid_patients_obs['days'] = np.around(covid_patients_obs.lab_days / np.timedelta64(1, 'D'))
covid_patients_obs['VALUE'] = pd.to_numeric(covid_patients_obs['VALUE'], errors='coerce')

In [ ]:
import matplotlib.ticker as ticker
loinc_to_display = {'CODE_y = 48065-7': 'D-dimer', 'CODE_y = 2276-4': 'Serum Ferritin',
                    'CODE_y = 89579-7': 'High Sensitivity Cardiac Troponin I',
                    'CODE_y = 26881-3': 'IL-6', 'CODE_y = 731-0': 'Lymphocytes',
                    'CODE_y = 14804-9': 'Lactate dehydrogenase'}
catplt = sns.catplot(x="days", y="VALUE", hue="survivor", kind="box", col='CODE_y',
            col_wrap=2, sharey=False, sharex=False, data=covid_patients_obs, palette=["C1", "C0"])

for axis in catplt.fig.axes:
    axis.xaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))
    axis.xaxis.set_major_locator(ticker.MultipleLocator(base=4))
    axis.set_title(loinc_to_display[axis.title.get_text()])

plt.show()

In [ ]:
loinc_to_display = {'CODE_y = 48065-7': 'D-dimer', 'CODE_y = 2276-4': 'Serum Ferritin',
                    'CODE_y = 89579-7': 'High Sensitivity Cardiac Troponin I',
                    'CODE_y = 26881-3': 'IL-6', 'CODE_y = 731-0': 'Lymphocytes',
                    'CODE_y = 14804-9': 'Lactate dehydrogenase'}
catplt = sns.catplot(x="days", y="VALUE", hue="survivor", kind="point", col='CODE_y',
            col_wrap=2, sharey=False, sharex=False, data=covid_patients_obs, palette=["C1", "C0"])

for axis in catplt.fig.axes:
    axis.xaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))
    axis.xaxis.set_major_locator(ticker.MultipleLocator(base=4))
    axis.set_title(loinc_to_display[axis.title.get_text()])

plt.show()

Set up a new DataFrame with boolean columns representing various outcomes, like admit, recovery or death

In [ ]:
# cp = covid_conditions.merge(patients, how='left', left_on='PATIENT', right_on='Id')
# isolation_ids = care_plans[(care_plans.CODE == 736376001) & (care_plans.REASONCODE == 840539006)].PATIENT
# cp['isolation'] = cp.Id.isin(isolation_ids)
# cp['admit'] = cp.Id.isin(inpatient_ids)
# cp['recovered'] = cp.Id.isin(survivor_ids)
# cp['death'] = cp.DEATHDATE.notna()
# icu_ids = encounters[encounters.CODE == 305351004].PATIENT
# cp['icu_admit'] = cp.Id.isin(icu_ids)
# vent_ids = procedures[procedures.CODE == 26763009].PATIENT
# cp['ventilated'] = cp.Id.isin(vent_ids)